Inferenza con i LLM Instruct: chat templates, few-shot learning e COT
======================================================


Questo notebook analizza l'uso dei modelli di tipo instruct, l'uso dei chat templates e due tipologie di tecniche di promting (few-shot e COT) con il modello [Qwen 3 4B](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507).

INSTALLAZIONE E IMPORT
============================================================================

Per eseguire questo notebook, installare prima le dipendenze:

pip install transformers torch bitsanbytes

Su google Colab sono già pre-installate, serve installare bitsandbytes per la quantizzazione

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# definiamo una funzione per fare inferenza ad un LLM

def ask(message, tokenizer, model, device):

    # Tokenizzazione: converte testo in tensori di ID
    inputs = tokenizer(
        message,
        return_tensors="pt"  # Restituisce tensori PyTorch
    )
    inputs['input_ids'] = inputs['input_ids'].to(device)
    inputs['attention_mask'] = inputs['attention_mask'].to(device)

    gen_config = GenerationConfig(
        do_sample=True,
        max_new_tokens=256,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id
    )

    response = model.generate(
        **inputs,
        generation_config=gen_config)

    answer_full = tokenizer.decode(response[0], skip_special_tokens=False)
    answer = tokenizer.decode(response[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

    return answer_full, answer

In [ ]:
device = 'cuda'

# --- Qwen 3.2 Instruct ---
print("Caricamento Qwen 3.2 4B Instruct...")
model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    trust_remote_code=True
)

answer_full, answer = ask(prompt, tokenizer, model, device)

print(answer_full)


Caricamento Qwen 3.2 4B Instruct...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Di seguito è riportata un'istruzione che descrive un'attività, abbinata a un input che fornisce ulteriori informazioni contestuali. Scrivi una risposta che completi in modo appropriato la richiesta.

### Istruzione:
Classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.

### Input:
Adoro la pasta.

### Risposta:
positivo

Spiegazione: La frase "Adoro la pasta" esprime un sentimento positivo, poiché l'uso della parola "adoro" indica un forte apprezzamento. Pertanto, il testo è classificabile come positivo. La risposta è pertanto "positivo". Questo è coerente con l'etichetta richiesta. La risposta finale è: positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. positivo. po

## Token speciali nei template

I token speciali sono marker che delimitano ruoli e messaggi.

Ogni modello usa token diversi per strutturare la conversazione.

In [ ]:
print(tokenizer.SPECIAL_TOKENS_ATTRIBUTES)

['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token']


In [ ]:
print("Token speciali:")
for attr_name in tokenizer.SPECIAL_TOKENS_ATTRIBUTES:
    token = getattr(tokenizer, attr_name, "N/A") # Usa getattr per accedere all'attributo
    print(f"  {attr_name}: {token}")

Token speciali:
  bos_token: None
  eos_token: <|im_end|>
  unk_token: None
  sep_token: None
  pad_token: <|im_end|>
  cls_token: None
  mask_token: None


I `Chat template` sono formati standardizzati che i modelli Instruct si aspettano.
Ogni modello ha il proprio formato memorizzato nel tokenizer.

Struttura base:
- `system`
    - Definisce il comportamento e le caratteristiche del modello
   - Imposta il contesto generale
   - Definisce personalità, stile, limitazioni
   - Esempio: "Sei un assistente utile e preciso"

- `user`
    - Rappresenta i messaggi dell'utente, ovvero domande o richieste
   - Input che richiede una risposta

- `assistant`
    - Rappresenta le risposte del modello
   - Usato per esempi di conversazioni precedenti
   - Durante l'inferenza, il modello genera questo ruolo

## Inferenze con un messaggio semplice

In [ ]:
# Creiamo un messaggio semplice
messages_simple = [
    {"role": "user", "content": "Qual è la capitale dell'Italia?"}
]

print("Messaggi input:")
print(messages_simple)

# Applicazione template
prompt_simple = tokenizer.apply_chat_template(
    messages_simple,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(prompt_simple)
print("\n" + "="*80)

Messaggi input:
[{'role': 'user', 'content': "Qual è la capitale dell'Italia?"}]
<|im_start|>user
Qual è la capitale dell'Italia?<|im_end|>
<|im_start|>assistant




In [ ]:
answer_full, answer = ask(prompt_simple, tokenizer, model, device)

print(answer_full)
print(f'\n\n*** Risposta generata -> {answer}')

<|im_start|>user
Qual è la capitale dell'Italia?<|im_end|>
<|im_start|>assistant
La capitale dell'Italia è **Roma**.<|im_end|>


*** Risposta generata -> La capitale dell'Italia è **Roma**.


## Inferenze con un messaggio contenente il ruolo system

In [ ]:
# Creiamo un messaggio con il ruolo system
messages_system = [
    {"role": "system", "content": "Sei un assistente esperto di geografia che risponde in modo conciso e preciso."},
    {"role": "user", "content": "Qual è la capitale dell'Italia?"}
]

print("Messaggi input:")
print(messages_system)

# Applicazione template
prompt_system = tokenizer.apply_chat_template(
    messages_system,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(prompt_system)

answer_full, answer = ask(prompt_system, tokenizer, model, device)

print(f'\n\n*** Risposta generata -> {answer}')

Messaggi input:
[{'role': 'system', 'content': 'Sei un assistente esperto di geografia che risponde in modo conciso e preciso.'}, {'role': 'user', 'content': "Qual è la capitale dell'Italia?"}]
<|im_start|>system
Sei un assistente esperto di geografia che risponde in modo conciso e preciso.<|im_end|>
<|im_start|>user
Qual è la capitale dell'Italia?<|im_end|>
<|im_start|>assistant



*** Risposta generata -> La capitale dell'Italia è Roma.


## Inferenze con un messaggio multi-turn

In [ ]:
# Creiamo un messaggio multi-turn
messages_conversation = [
    {"role": "system", "content": "Sei un tutor di matematica paziente e chiaro."},
    {"role": "user", "content": "Cos'è il teorema di Pitagora?"},
    {"role": "assistant", "content": "Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²"},
    {"role": "user", "content": "Puoi farmi un esempio pratico?"}
]

print("Messaggi input:")
print(messages_conversation)

# Applicazione template
prompt_conversation = tokenizer.apply_chat_template(
    messages_conversation,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(prompt_conversation)

answer_full, answer = ask(prompt_conversation, tokenizer, model, device)

print(f'\n\n*** Risposta generata -> {answer}')

Messaggi input:
[{'role': 'system', 'content': 'Sei un tutor di matematica paziente e chiaro.'}, {'role': 'user', 'content': "Cos'è il teorema di Pitagora?"}, {'role': 'assistant', 'content': "Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²"}, {'role': 'user', 'content': 'Puoi farmi un esempio pratico?'}]
<|im_start|>system
Sei un tutor di matematica paziente e chiaro.<|im_end|>
<|im_start|>user
Cos'è il teorema di Pitagora?<|im_end|>
<|im_start|>assistant
Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²<|im_end|>
<|im_start|>user
Puoi farmi un esempio pratico?<|im_end|>
<|im_start|>assistant



*** Risposta generata -> Certo! Ecco un esempio pratico del **teorema di Pitagora**.

---

### 📐 Problema:
Supponiamo che abbia un triangolo rettangolo (cioè con un angolo di 90°).  
I due lati che 

## Few-shot prompting

Tecnica di prompting in cui insieme all'istruzione viene fornito uno o più esempi.

Utile al modello per apprendere il task e la struttura dell'output.

In [ ]:
# Creiamo un messaggio per il few-shot prompting
few_shot_messages = [
    {"role": "system", "content": "Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione."},
    {"role": "user", "content": "Potresti dirmi a che ora parte il prossimo treno per Roma?"},
    {"role": "assistant", "content": "Could you tell me what time the next train to Rome leaves"},
    {"role": "user", "content": "Stiamo ascoltando un vecchio disco."},
    {"role": "assistant", "content": "We are listening to an old record."},
    {"role": "user", "content": "Ieri ho studiato per l'esame di venerdì, spero di passarlo."}
    # risposta attesa -> Yesterday I studied for Friday exam, I hope that I will pass it.
]

print("Messaggi input:")
print(few_shot_messages)

# Applicazione template
prompt_few_shot = tokenizer.apply_chat_template(
    few_shot_messages,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(prompt_few_shot)

answer_full, answer = ask(prompt_few_shot, tokenizer, model, device)

print(f'\n\n*** Risposta generata -> {answer}')

Messaggi input:
[{'role': 'system', 'content': "Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione."}, {'role': 'user', 'content': 'Potresti dirmi a che ora parte il prossimo treno per Roma?'}, {'role': 'assistant', 'content': 'Could you tell me what time the next train to Rome leaves'}, {'role': 'user', 'content': 'Stiamo ascoltando un vecchio disco.'}, {'role': 'assistant', 'content': 'We are listening to an old record.'}, {'role': 'user', 'content': "Ieri ho studiato per l'esame di venerdì, spero di passarlo."}]
<|im_start|>system
Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione.<|im_end|>
<|im_start|>user
Potresti dirmi a che ora parte il prossimo treno per Roma?<|im_end|>
<|im_start|>assistant
Could you tell me what time the next train to Rome leaves<|im_end|>
<|im_start|>user
Stiamo ascoltando un vecchio disco.<|im_end|>
<|im_start|>assistant
We are listening to an old record.<|im_end|>
<|im_start|>user
Ieri ho studiato per l'

## Chain of Thoughts

Tecnica di prompting in cui viene chiesto al modello di *ragionare* e restituire la sua predizione e i passaggi logici necessari per giungere alla predizione.

Utile al modello per i task più complessi.

In [ ]:
# Creiamo un messaggio per il cot, chiederemo al modello di classificare un testo dal data set BBC-text
# Multiclass dataset of BBC news divided in 5 categories with reference to their topic.
# [Dataset reference](https://www.kaggle.com/datasets/yufengdev/bbc-fulltext-and-category/code)

istruzione = """You are a text classifier. Follow these steps:\n
1. Identify key topics and entities in the article\n
2. Consider which category best matches these topics\n
3. Provide the category: business, politics, tech, sport, or entertainment\n

Format your response as:\n
Reasoning: [your analysis]\n
Category: [category name]\n"""

text = "ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and festive comedy christmas with the kranks.  ocean s twelve box office triumph marks the fourth-biggest opening for a december release in the us  after the three films in the lord of the rings trilogy. the sequel narrowly beat its 2001 predecessor  ocean s eleven which took $38.1m (£19.8m) on its opening weekend and $184m (£95.8m) in total. a remake of the 1960s film  starring frank sinatra and the rat pack  ocean s eleven was directed by oscar-winning director steven soderbergh. soderbergh returns to direct the hit sequel which reunites clooney  pitt and roberts with matt damon  andy garcia and elliott gould. catherine zeta-jones joins the all-star cast.  it s just a fun  good holiday movie   said dan fellman  president of distribution at warner bros. however  us critics were less complimentary about the $110m (£57.2m) project  with the los angeles times labelling it a  dispiriting vanity project . a milder review in the new york times dubbed the sequel  unabashedly trivial ."
# entertainment è l'etichetta attesa

cot_messages = [
    {"role": "system", "content": istruzione},
    {"role": "user", "content": text},
]

print("Messaggi input:")
print(cot_messages)

# Applicazione template
prompt_cot = tokenizer.apply_chat_template(
    cot_messages,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(prompt_cot)

answer_full, answer = ask(prompt_cot, tokenizer, model, device)

print(f'\n\n*** Risposta generata -> {answer}')

Messaggi input:
[{'role': 'system', 'content': 'You are a text classifier. Follow these steps:\n\n1. Identify key topics and entities in the article\n\n2. Consider which category best matches these topics\n\n3. Provide the category: business, politics, tech, sport, or entertainment\n\n\nFormat your response as:\n\nReasoning: [your analysis]\n\nCategory: [category name]\n'}, {'role': 'user', 'content': 'ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and fest